In [0]:
%sql
-- ============================================================
-- Create a metric view in retail_bootcamp.gold with 5 KPIs
-- 1. Net Revenue          → SUM(net_sales) from fact_order_line
-- 2. Gross Margin %       → SUM(gross_profit) / SUM(net_sales)
-- 3. Online Conversion % → SUM(orders) / SUM(sessions) from fact_web_session
-- 4. Return Rate %        → SUM(returns) / SUM(order_lines)
-- 5. Stock-out Rate %     → SUM(stock_out_days) / SUM(inventory_snapshots)
-- ============================================================
CREATE OR REPLACE VIEW retail_bootcamp.gold.kpi_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: >
    WITH fin AS (
      SELECT to_date(CAST(month_date_key AS STRING), 'yyyyMMdd') AS month_date,
             store_id
      FROM retail_bootcamp.gold.fact_finance_monthly
    ),
    order_agg AS (
      SELECT DATE_TRUNC('MONTH', to_date(CAST(date_key AS STRING), 'yyyyMMdd')) AS month_date,
             store_id,
             SUM(net_sales) AS net_sales,
             SUM(gross_profit) AS gross_profit,
             COUNT(DISTINCT order_item_id) AS order_lines
      FROM retail_bootcamp.gold.fact_order_line
      GROUP BY 1, 2
    ),
    returns_agg AS (
      SELECT DATE_TRUNC('MONTH', to_date(CAST(date_key AS STRING), 'yyyyMMdd')) AS month_date,
             store_id,
             COUNT(DISTINCT return_id) AS returns_count
      FROM retail_bootcamp.gold.fact_returns
      GROUP BY 1, 2
    ),
    web_session_agg AS (
      SELECT DATE_TRUNC('MONTH', to_date(CAST(date_key AS STRING), 'yyyyMMdd')) AS month_date,
             'ON01' AS store_id,
             COUNT(*) AS web_sessions,
             SUM(CASE WHEN ordered_flag THEN 1 ELSE 0 END) AS web_orders
      FROM retail_bootcamp.gold.fact_web_session
      GROUP BY 1, 2
    ),
    inventory_agg AS (
      SELECT DATE_TRUNC('MONTH', to_date(CAST(date_key AS STRING), 'yyyyMMdd')) AS month_date,
             store_id,
             COUNT(*) AS inv_snapshots,
             SUM(CASE WHEN is_stock_out THEN 1 ELSE 0 END) AS stock_out_days
      FROM retail_bootcamp.gold.fact_inventory_daily
      GROUP BY 1, 2
    )
    SELECT fin.month_date,
           fin.store_id,
           COALESCE(oa.net_sales, 0)        AS net_sales,
           COALESCE(oa.gross_profit, 0)     AS gross_profit,
           COALESCE(oa.order_lines, 0)      AS order_lines,
           COALESCE(ra.returns_count, 0)   AS returns_count,
           COALESCE(ws.web_sessions, 0)    AS web_sessions,
           COALESCE(ws.web_orders, 0)      AS web_orders,
           COALESCE(ia.inv_snapshots, 0)   AS inv_snapshots,
           COALESCE(ia.stock_out_days, 0)  AS stock_out_days,
           ds.store_name                    AS store_name,
           ds.channel                       AS channel
    FROM fin
    LEFT JOIN order_agg      oa ON fin.month_date = oa.month_date AND fin.store_id = oa.store_id
    LEFT JOIN returns_agg    ra ON fin.month_date = ra.month_date AND fin.store_id = ra.store_id
    LEFT JOIN web_session_agg ws ON fin.month_date = ws.month_date AND fin.store_id = ws.store_id
    LEFT JOIN inventory_agg   ia ON fin.month_date = ia.month_date AND fin.store_id = ia.store_id
    LEFT JOIN retail_bootcamp.gold.dim_store ds ON fin.store_id = ds.store_id
  dimensions:
    - name: Month
      expr: month_date
      display_name: Month
      comment: First day of the month
      format:
        type: date
        date_format: locale_short_month
      synonyms:
        - month_date
        - period
    - name: Store
      expr: store_id
      display_name: Store
      comment: Store identifier (ON01 = online)
      synonyms:
        - store_id
    - name: Store Name
      expr: store_name
      display_name: Store Name
      comment: Human-readable store name from dim_store
    - name: Channel
      expr: channel
      display_name: Channel
      comment: Sales channel (Online or Physical)
      synonyms:
        - store_type
  measures:
    - name: Net Revenue
      expr: SUM(net_sales)
      display_name: Net Revenue
      comment: Total net sales after discounts, EUR
      format:
        type: currency
        currency_code: EUR
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - revenue
        - sales
        - net sales
    - name: Gross Margin Pct
      expr: SUM(gross_profit) / NULLIF(SUM(net_sales), 0)
      display_name: Gross Margin %
      comment: Gross profit as a fraction of net sales (0-1 ratio)
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - gross margin
        - margin
        - profitability
    - name: Online Conversion Rate Pct
      expr: SUM(web_orders) / NULLIF(SUM(web_sessions), 0)
      display_name: Online Conversion Rate %
      comment: Web sessions that ended in an order as a fraction of all sessions (0-1 ratio). Only ON01 has web data.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - conversion rate
        - conversion
        - web conversion
    - name: Return Rate Pct
      expr: SUM(returns_count) / NULLIF(SUM(order_lines), 0)
      display_name: Return Rate %
      comment: Returned order lines as a fraction of all order lines (0-1 ratio)
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - return rate
        - returns
    - name: Stock Out Rate Pct
      expr: SUM(stock_out_days) / NULLIF(SUM(inv_snapshots), 0)
      display_name: Stock-out Rate %
      comment: Inventory snapshots with zero available stock as a fraction of all snapshots (0-1 ratio)
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - stock out
        - stockout
        - out of stock
$$

In [0]:
%sql
-- Query the metric view: all 5 KPIs by Channel
SELECT
  `Channel`,
  MEASURE(`Net Revenue`) AS `net_revenue`,
  MEASURE(`Gross Margin Pct`) AS `gross_margin_pct`,
  MEASURE(`Online Conversion Rate Pct`) AS `online_conversion_pct`,
  MEASURE(`Return Rate Pct`) AS `return_rate_pct`,
  MEASURE(`Stock Out Rate Pct`) AS `stock_out_pct`
FROM retail_bootcamp.gold.kpi_metrics
GROUP BY ALL
ORDER BY `net_revenue` DESC

Channel,net_revenue,gross_margin_pct,online_conversion_pct,return_rate_pct,stock_out_pct
Store,443301.45,0.405858,null,0.06463595839524518,0.02548148148148148
Online,207500.10,0.413933,0.0862,0.11180124223602485,0.014074074074074074


In [0]:
%sql
-- Query the metric view: all 5 KPIs by Month (trend analysis)
SELECT
  `Month`,
  MEASURE(`Net Revenue`) AS `net_revenue`,
  MEASURE(`Gross Margin Pct`) AS `gross_margin_pct`,
  MEASURE(`Online Conversion Rate Pct`) AS `online_conversion_pct`,
  MEASURE(`Return Rate Pct`) AS `return_rate_pct`,
  MEASURE(`Stock Out Rate Pct`) AS `stock_out_pct`
FROM retail_bootcamp.gold.kpi_metrics
GROUP BY ALL
ORDER BY `Month` DESC
LIMIT 12

Month,net_revenue,gross_margin_pct,online_conversion_pct,return_rate_pct,stock_out_pct
2026-09-01,25017.00,0.408562,0.08396946564885496,0.11864406779661017,0.023434343434343436
2026-08-01,20740.00,0.425796,0.11333333333333333,0.14925373134328357,0.026588465298142717
2026-07-01,27501.65,0.363129,0.08906882591093117,0.03296703296703297,0.023197492163009405
2026-06-01,22825.10,0.361887,0.07386363636363637,0.0821917808219178,null
2026-05-01,15649.00,0.405521,0.07352941176470588,0.09090909090909091,null
2026-04-01,18717.00,0.413742,0.06382978723404255,0.03773584905660377,null
2026-03-01,26625.00,0.413521,0.09202453987730061,0.078125,null
2026-02-01,10329.00,0.477394,0.07317073170731707,0.1724137931034483,null
2026-01-01,8420.00,0.474941,0.02727272727272727,0.1111111111111111,null
2025-12-01,36271.00,0.459761,0.09965635738831616,0.09420289855072464,null
